### <b>Phase 5: Model Evaluation and Improvement</b>

#### <b>Objective</b>

The objective of this phase is to improve the quality of movie recommendations by enhancing the feature engineering and similarity calculation process.

#### <b>Improvements Implemented</b>

##### <b>Weighted Features</b>

Different metadata fields contribute differently to movie similarity. Therefore, weighted importance was assigned:

- Genre → Highest Weight
- Director → High Weight
- Actors → High Weight
- Writer → Medium Weight
- Plot → Medium Weight

##### <b>Text Normalization</b>

All text was converted to lowercase to ensure consistency.

##### <b>Stemming</b>

Porter Stemmer was applied to reduce words to their root form.

Examples:

- running → run
- runs → run
- played → play

This reduces vocabulary size and improves similarity detection.

##### <b>Enhanced Ranking</b>

Recommendations are ranked using:

Final Score = 0.85 × Similarity Score + 0.15 × Normalized IMDb Rating

This prioritizes highly similar content while also promoting well-rated movies.


#### Step 1 : Import Libraries

In [2]:
# Import Libraries
import pandas as pd
import numpy as np

from nltk.stem.porter import PorterStemmer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

#### Step 2 : Load the Dataset

In [3]:
# Load Original Dataset
df = pd.read_csv("../data/Netflix_data.csv")

print(df.shape)

(3516, 17)


#### Step 3 : Select Required Columns

In [4]:
# Select Required Columns
movies = df[
    [
        "imdbID",
        "Title",
        "Genre",
        "Director",
        "Writer",
        "Actors",
        "Plot",
        "imdbRating"
    ]
].copy()

#### Step 4 : Handle Missing Values

In [5]:
# Handle Missing Values
text_columns = [
    "Genre",
    "Director",
    "Writer",
    "Actors",
    "Plot"
]

for col in text_columns:
    movies[col] = movies[col].fillna("")

#### Step 5 : Create and Combine Weighted Feature

In [6]:
# Create Weighted Feature
movies["Genre_w"] = movies["Genre"] * 3
movies["Director_w"] = movies["Director"] * 2
movies["Actors_w"] = movies["Actors"] * 2
movies["Writer_w"] = movies["Writer"]
movies["Plot_w"] = movies["Plot"]

In [7]:
# Combine Weighted Features
movies["tags"] = (
    movies["Genre_w"] + " " +
    movies["Director_w"] + " " +
    movies["Actors_w"] + " " +
    movies["Writer_w"] + " " +
    movies["Plot_w"]
)

#### Step 6 : Convert tags to lowercase

In [8]:
# Convert to lowercase
movies["tags"] = movies["tags"].str.lower()

#### Step 7 : Initialize Stemmer and Create Stemming Funtion

In [9]:
# Initialize stemmer
ps = PorterStemmer()

In [10]:
# Create Stemming Function
def stem_text(text):

    words = text.split()

    stemmed_words = [
        ps.stem(word)
        for word in words
    ]

    return " ".join(stemmed_words)

In [11]:
# Apply Stemming
movies["tags"] = movies["tags"].apply(
    stem_text
)

#### Step 8 : TF-IDF Vectorization

In [12]:
# TF-IDF Vectorization
tfidf = TfidfVectorizer(
    max_features=15000,
    stop_words="english"
)

tfidf_matrix = tfidf.fit_transform(
    movies["tags"]
)

In [13]:
# Compute SImilarity Matrix
cosine_sim = cosine_similarity(
    tfidf_matrix
)

#### Step 9 : Normalize Ratings

In [14]:
# Normalize Ratings
movies["imdbRating"] = pd.to_numeric(
    movies["imdbRating"],
    errors="coerce"
)

In [15]:
movies["imdbRating"] = movies[
    "imdbRating"
].fillna(
    movies["imdbRating"].median()
)

In [16]:
movies["rating_norm"] = (
    movies["imdbRating"]
    -
    movies["imdbRating"].min()
) / (
    movies["imdbRating"].max()
    -
    movies["imdbRating"].min()
)

#### Step 10 : Recommend Function

In [17]:
# Improved Recommend Function
def recommend(movie_title, top_n=10):

    matches = movies[
        movies["Title"].str.lower()
        ==
        movie_title.lower()
    ]

    if len(matches) == 0:
        return "Movie not found"

    idx = matches.index[0]

    scores = []

    for i, similarity in enumerate(
        cosine_sim[idx]
    ):

        rating = movies.iloc[i][
            "rating_norm"
        ]

        final_score = (
            0.85 * similarity
            +
            0.15 * rating
        )

        scores.append(
            (
                i,
                final_score
            )
        )

    scores = sorted(
        scores,
        key=lambda x: x[1],
        reverse=True
    )

    scores = scores[1:top_n+1]

    movie_ids = [
        i[0]
        for i in scores
    ]

    return movies.iloc[
        movie_ids
    ][
        [
            "Title",
            "imdbRating"
        ]
    ]

In [19]:
# Evaluate Recommendation
recommend("Batman")

,Title,imdbRating
1252,Batman Returns,7.1
2009,Misty Ride - Jack C. Mancino - Balogh Csaba bi...,6.6
1071,Batman: Mask of the Phantasm,7.8
1041,Batman Beyond,8.1
1117,Batman Beyond: Return of the Joker,7.8
1381,"Batman: The Dark Knight Returns, Part 2",8.4
1091,The Batman,7.4
1383,Batman: Under the Red Hood,8.1
1534,Batman Forever,5.5
1434,Batman: Black and White,7.6


#### Step 11 :  Save All the models

In [20]:
# Save Improved Model
import pickle
import os

os.makedirs(
    "../models",
    exist_ok=True
)

with open(
    "../models/tfidf_improved.pkl",
    "wb"
) as f:

    pickle.dump(
        tfidf,
        f
    )


In [21]:
with open(
    "../models/cosine_similarity_improved.pkl",
    "wb"
) as f:

    pickle.dump(
        cosine_sim,
        f
    )


In [22]:
movies.to_pickle(
    "../models/movies_improved.pkl"
)

#### Outcome

The improved model generates more accurate and relevant recommendations by combining content similarity with movie quality indicators.